## RAG, step 1: the idea

**Problem:** the model does not know our documents and may invent an answer.

**Idea:** an open-book exam. Find the right page first, then answer from it.

| Part | Job | Uses the model? |
|---|---|---|
| 1. Keep | A small library of documents | No |
| 2. Find | Pick the best matching document(s) for the question | No (or a small embedding model) |
| 3. Answer | Give the model the question and the document; it answers from it | Yes |

**Why it gives a source:** we know which document we handed over.

**Keyword search by hand (documents D1 to D4, invented):**

| Question | Right note | Shared-word scores | Result |
|---|---|---|---|
| Which part carries current between breakers in a panel? | D2 | D2 scores 5, the others 1 or 0 | Works |
| How do I close the gap where a wire goes into a box? | D3 | All score 0 | Fails: different words, same meaning |

**Lesson:** matching on words breaks when the question uses different words. Step 2 compares meaning with embeddings.

## RAG, step 2a: embeddings

**Problem (from step 1):** matching shared words fails when the question uses different words for the same thing.

**Idea:** turn every sentence into a list of numbers that stands for its meaning. Sentences with similar meaning get similar lists. The computer then compares lists, not words.

| Word-matching | Embedding |
|---|---|
| Counts shared words | Compares meaning |
| "gap where a wire goes into a box" and "cable glands seal the point where a cable enters an enclosure" share almost no words, so the score is 0 | The two lists end up close together, so they match |

**A map picture:** think of a map of meaning. Every sentence is a pin. Sentences about the same thing land close together. A question is also dropped on the map, and the nearest pins are the best documents.

**Which model does this?** A separate, small model (`nomic-embed-text`). It does not write text. It only turns text into numbers.

**Next:** step 2b builds the first vector. Step 2c compares two vectors.


## RAG, step 2b: my first vector

| Item | My result |
|---|---|
| Embedding model | `nomic-embed-text` (small, separate from the text model) |
| Sentence | "Cable glands seal the point where a cable enters an enclosure." |
| Vector length | 768 |
| First five numbers | 0.0307, 0.0535, -0.1459, -0.0556, 0.0245 |
| My guess for the length | (fill in yours) |

| | Text model (`gemma4:e2b`) | Embedding model (`nomic-embed-text`) |
|---|---|---|
| Job | Writes text | Turns text into a position |
| Call | `ollama.chat(...)` | `ollama.embed(...)` |
| Result | `response.message.content` | `response.embeddings` (a list of vectors) |

```
response
└── embeddings    a list: one vector per input text
    └── [0]       the first vector, 768 numbers
```

**Reading it:** each number alone means nothing. The whole list, compared with other lists, carries the meaning.

## RAG, step 2c: compare two sentences

**Problem:** two sentences are now two lists of 768 numbers. We need one score for "how close in meaning".

**Idea:** cosine similarity. It compares the direction of the two lists and ignores their size.

| Code | Meaning |
|---|---|
| `ollama.embed(...)` | The action. Sends the sentences to the embedding model. |
| `.embeddings` | Not an action. It reads the list of vectors out of the reply, like `response.message.content` in a chat call. |
| `vectors[0]` | The vector of the first sentence |
| `a`, `b` | The two vectors being compared |
| `x`, `y` | One number from `a` and the matching number from `b` |
| `zip(a, b)` | Pairs the numbers up: first with first, second with second |
| `dot` | Multiply each pair, then add them up |
| `length_a`, `length_b` | The size of each vector: square every number, add them, take the square root |

**The formula in plain words:**

Score = dot ÷ (length_a × length_b)

**Tiny example (2 numbers each):**

| a | b | dot | length_a | length_b | Score |
|---|---|---|---|---|---|
| [1, 2] | [2, 4] | 10 | 2.236 | 4.472 | 1.0 (same direction) |
| [1, 0] | [0, 1] | 0 | 1 | 1 | 0 (unrelated) |

**How to read a score:** close to 1 means similar meaning. Close to 0 means unrelated.

**Mistake I made:** the function worked out `dot` and the two lengths but had no `return` line. It gave back `None`, and `round(None, 3)` raised a `TypeError`. A function without `return` hands back nothing.

**My results:**

| Pair | Sentences | Score |
|---|---|---|
| 0 vs 1 | Cable glands note and the question about the gap where a wire goes into a box | (fill in) |
| 0 vs 2 | Cable glands note and the busbar note | (fill in) |
| 1 vs 2 | The question and the busbar note | (fill in) |

**My guess for the highest pair:** (fill in)

**My results:**

| Pair | Sentences | Score |
|---|---|---|
| 0 vs 1 | Cable glands note and the question about the gap where a wire goes into a box | **0.574** |
| 0 vs 2 | Cable glands note and the busbar note | 0.495 |
| 1 vs 2 | The question and the busbar note | 0.532 |

**Reading it:** the pair with the same meaning but almost no shared words scored highest. This is the case where word matching scored 0. The gaps are small (0.495 to 0.574), because all three sentences are about electrical panels. Scores are only useful when compared with each other, so the next step ranks several notes for one question.

**Check on the maths:** `cosine_verbose([1, 2, 3], [3, 2, 1])` gave dot = 10, both lengths 3.742, score 0.714. This matches the hand calculation 10 ÷ 14.

## RAG, step 3: rank four notes for a question

**Problem:** find the note that best matches a question, using only meaning.

**Method:** embed the four invented notes (D1 to D4) once, embed the question, score each note with cosine similarity, and sort the scores. The top note is the one we hand to the model.

| Note | Content |
|---|---|
| D1 | Moulded case circuit breaker switches off a circuit when the current is too high |
| D2 | Busbar carries current between circuit breakers in a panel |
| D3 | Cable glands seal the point where a cable enters an enclosure |
| D4 | Current transformer measures current in a cable so a meter can read it |

**Results (top score, and the gap to second place):**

| Question | `nomic-embed-text` (768 numbers) | `mxbai-embed-large` (1024 numbers) |
|---|---|---|
| Hard: "How do I close the gap where a wire goes into a box?" (right note: D3) | Wrong. D2 first (0.583), D3 third (0.581). Gap 0.002. | Right. D3 first (0.699). Gap 0.122. |
| Easy: "Which part carries current between breakers in a panel?" (right note: D2) | Right. D2 (0.833). Gap 0.163. | Right. D2 (0.858). Gap 0.155. |

**Findings:**

1. The question with different words from the note was the real test. The smaller model failed it. The larger model passed it.
2. The task labels (`search_query:`, `search_document:`) for `nomic-embed-text` did not fix the hard question. Scores for all four notes moved close together.
3. A question that shares words with the note works with either model.
4. Score gaps matter more than scores. A gap of 0.002 is a coin flip. A gap of 0.12 is a clear choice.
5. Notes and question must be embedded by the same model.

**Limits:** four notes and two questions is a small test. It shows a difference between the models. It does not measure how often retrieval is right.

**Mistakes made on the way:**

| Mistake | Effect |
|---|---|
| Pair stored as `(note_id, score)` | Sorted by the id, and `round()` failed on text |
| `.embeddings[0]` on the notes | Kept only the first note's vector |
| One name for the list and the loop variable | The list was overwritten |

## RAG, steps 4 and 5: answer from the note, show the source, refuse when needed

**Problem:** finding the right note is not enough. The model must answer from that note and say where the answer came from.

**Method:**

1. Take the top note from the ranking.
2. Send the model two messages. The system message sets the rule: "use only the note; if it is not there, say you cannot answer". The user message holds the note and the question.
3. Print the answer. Our code prints the source id, not the model.

**Why our code prints the source:** we know which note we handed over, so the source cannot be invented by the model.

| Part | Code | Job |
|---|---|---|
| Best match | `scores[0]` | Top pair `(score, note id)` |
| Rule | system message | Keeps the model inside the note |
| Material | user message with `Note:` and `Question:` | The note and the question |
| Source | `print(f"Source: {best_id}")` | Printed by our code |

**Packing it into a function:** `answer_from_notes(question)` returns three values: the answer, the source (or `None`), and the best score. A `for` loop then asks a list of questions.

**Results (`mxbai-embed-large` for search, `gemma4:e2b` for answers):**

| Question | Answer | Source | Score |
|---|---|---|---|
| Which part carries current between breakers in a panel? | Busbar carries current between breakers | D2 | 0.858 |
| How do I close the gap where a wire goes into a box? | Cable glands seal where a cable enters an enclosure | D3 | 0.699 |
| What is the price of a busbar? | I cannot answer from the notes. | none | 0.609 |
| What lets a meter read the current in a cable? | Current transformer | D4 | 0.913 |

**The refusal problem:** retrieval always returns the nearest note, even when no note fits. For the price question, D2 was nearest (0.609), but it holds no price. Printing `Source: D2` would suggest D2 supported the answer. The code now reports `Source: none` when the model refuses.

**Limits:**

1. The refusal check looks for the words "cannot answer". If the model rewords its refusal, the check misses it.
2. Four questions are a demonstration, not a measurement.
3. The weakest real match (0.699) is close to the refusal score (0.609). A score cut-off would be fragile.
4. The model sometimes repeats the note instead of rephrasing it. That is acceptable here, because the answer stays faithful.

**Mistakes made on the way:**

| Mistake | Effect |
|---|---|
| Indented line inside a function body (one level too deep) | `IndentationError` |
| Old `return` line left under the new one | Dead code that never runs |
| Old `print` left beside the new `if` | Source printed twice |

## RAG, step 6: following one question through the project

**Idea:** when a question is sent to `agent.run_agent(...)`, it passes through several files and two different models. This note maps that journey.

**The journey:**

```
Cell 31 (notebook)
  agent.run_agent("question")
        │
        ▼
agent.py  run_agent
  messages = [question]        steps = []
        │
        ▼   ┌───────────── loop: up to 5 rounds ─────────────┐
        │   │                                                 │
        ▼   │                                                 │
  ┌─────────────────────────────────────────┐                 │
  │ CHAT MODEL  (gemma4:e2b)                │                 │
  │ ollama.chat(messages + tool list)       │                 │
  │ reads the question and tool descriptions│                 │
  └───────────┬──────────────────┬──────────┘                 │
      asks for a tool       writes text                        │
              │                  │                             │
              ▼                  │                             │
  agent.py  run_tool(name, args) │                             │
   looks the name up in TOOLS    │                             │
        │                        │                             │
        ├─ lookup_price ──► tools.py   PRICES dict ──► "95.0"  │
        ├─ list_parts ────► tools.py   joins the names         │
        ├─ line_total ────► tools.py   price × quantity        │
        │                                                      │
        └─ search_notes ──► retrieval.py                       │
               │                                               │
               ▼                                               │
           rank_notes(question)                                │
               │                                               │
               ├─► EMBED MODEL (mxbai-embed-large)             │
               │   ollama.embed: the 4 notes → 4 vectors       │
               ├─► EMBED MODEL                                 │
               │   ollama.embed: the question → 1 vector       │
               ├─► cosine_notes(question, note)  ×4            │
               └─► sort, take the best                         │
               │                                               │
               ▼                                               │
           "Source D2 (score 0.861): A busbar carries ..."     │
        │                                                      │
        ▼                                                      │
  schemas.py  ToolStep(name, arguments, result)  ──► steps     │
  the result goes back to messages as role "tool" ─────────────┘
                                 │
                                 ▼  (the model wrote text)
agent.py  build_result(question, answer, steps)
        ├─ find_sources(steps)         reads the "Source D2" ids
        ├─ checks.py  unsupported_numbers(question, result)
        │      └─ find_numbers(text)   regex finds the numbers
        │      allowed numbers = question + tool results + tool inputs
        │      unsupported = numbers in the answer not in "allowed"
        ▼
schemas.py  AgentResult(answer, steps, sources, unsupported)
        │
        ▼
Cell 31 prints answer, sources, unsupported
```

**Who does what:**

| File | Function | Job |
|---|---|---|
| `agent.py` | `run_agent` | The loop. Keeps the message history, calls the chat model, runs requested tools |
| `agent.py` | `run_tool` | Looks up a tool by name, runs it safely, returns text |
| `agent.py` | `find_sources`, `build_result` | Build the final result with sources and the number check |
| `tools.py` | `lookup_price`, `list_parts`, `line_total` | Price data and arithmetic |
| `retrieval.py` | `search_notes`, `rank_notes`, `cosine_notes` | Find the best note by meaning |
| `checks.py` | `find_numbers`, `unsupported_numbers` | Flag numbers the evidence does not back |
| `schemas.py` | `ToolStep`, `AgentResult` | The shape of a step and of a run |

**When each model is used:**

| Model | Used when | How often |
|---|---|---|
| Chat (`gemma4:e2b`) | Every round of the loop. It chooses a tool or writes the answer. | One call per round |
| Embedding (`mxbai-embed-large`) | Only inside `search_notes`. It never writes text. | Two calls per search: the notes, then the question |

**Counts from my three Cell 31 runs:**

| Question | Chat calls | Embedding calls | Sources | Unsupported numbers |
|---|---|---|---|---|
| Price of one busbar | 2 | 0 | none | none |
| Cost of 3 busbars | 3 | 0 | none | none |
| Which part carries current | 2 | 2 | D2 | none |

**One detail from the log:** `search_notes` receives the question as the chat model rewrote it (`'which part carries current between breakers in a panel'`, in lowercase with no `?`). That is why the score was 0.861 in the agent run and 0.858 in the notebook run.

**How the number check works:** a number in the answer is allowed if it appears in the question, in a tool result, or in a tool input. Any other number is listed in `unsupported`. An empty list means every number has evidence behind it.

**Limits of the number check:**

1. It sees digits only. A number written as a word ("three") is not checked.
2. It checks that a number exists in the evidence, not that it is used correctly.
3. It cannot judge facts without numbers.